# Model 2B, Logistic Regression Experiment 03: Combined 2019 and 2023 Training

This append-only post-test sensitivity experiment copies the frozen Logistic Regression 2B-02 design. It changes only the training population: the model is fit on combined 2019 and 2023 rows instead of 2019 alone. The 28-field allowlist, model-side transformations, classifier settings, and 0.39 operating threshold remain unchanged.

The 2024 outcomes were already examined in the official final evaluation. This experiment therefore cannot replace that result or select a new model. It measures whether routine retraining with the completed 2023 development year changes the frozen design's 2024 performance.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

AIRPORT = "JFK"
TRAIN_YEARS = (2019, 2023)
TEST_YEAR = 2024
TARGET = "ArrDel15"
FROZEN_THRESHOLD = 0.39
RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.0


## Load and validate the arrival datasets

The 2019 and 2023 datasets are validated separately before concatenation. The 2024 dataset contains the same arrival target population used by Models 2A, 2B, and 2C.

In [2]:
def find_project_root(start: Path) -> Path:
    required = [
        Path("data/features") / f"{AIRPORT}_{year}_arrivals.csv"
        for year in (*TRAIN_YEARS, TEST_YEAR)
    ]
    for candidate in (start, *start.parents):
        if all((candidate / path).is_file() for path in required):
            return candidate
    raise FileNotFoundError(f"Could not locate required datasets: {required}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_DIR = PROJECT_ROOT / "data/features"
years = (*TRAIN_YEARS, TEST_YEAR)
source_frames = {
    year: pd.read_csv(FEATURE_DIR / f"{AIRPORT}_{year}_arrivals.csv", low_memory=False)
    for year in years
}


def validate(frame, year):
    frame["FlightDate"] = pd.to_datetime(frame["FlightDate"], errors="raise")
    assert frame["FlightDate"].dt.year.eq(year).all()
    assert frame["Dest"].eq(AIRPORT).all()
    assert frame[TARGET].notna().all()
    assert set(frame[TARGET].unique()).issubset({0, 0.0, 1, 1.0})
    return {"year": year, "rows": len(frame), "delay_rate": frame[TARGET].mean()}


pd.DataFrame([validate(source_frames[year], year) for year in years]).set_index("year")

,rows,delay_rate
year,,
2019,107354,0.2029
2023,109947,0.2463
2024,104555,0.2161


## Fit the unchanged Logistic Regression 2B-02 design

The combined training rows remain in chronological order. The original model search and threshold selection are not repeated.

In [3]:
CATEGORICAL_FEATURES = ["Reporting_Airline", "Origin"]
NUMERIC_FEATURES = ['SCHED_DEP_TIME_SIN', 'SCHED_DEP_TIME_COS', 'SCHED_ARR_TIME_SIN', 'SCHED_ARR_TIME_COS', 'DAY_OF_WEEK_SIN', 'DAY_OF_WEEK_COS', 'DAY_OF_YEAR_SIN', 'DAY_OF_YEAR_COS', 'IS_WEEKEND', 'CRSElapsedTime', 'LOG_DISTANCE', 'SCHEDULED_SPEED_PROXY', 'ASPM_THREE_HOUR_SCHEDULED_DEPARTURES', 'ASPM_THREE_HOUR_SCHEDULED_ARRIVALS', 'ASPM_CURRENT_MINUS_PREVIOUS_TRAFFIC', 'ASPM_NEXT_MINUS_CURRENT_TRAFFIC', 'ASPM_MAX_HOURLY_TRAFFIC', 'HourlyDryBulbTemperature', 'TEMP_DEWPOINT_SPREAD', 'LOG_PRECIPITATION', 'HourlyVisibility', 'WindX', 'WindY', 'ADVERSE_WEATHER', 'NOAA_AGE_MINUTES', 'DepDelay']
MARGIN_FEATURE = 'MINUTES_TO_SCHEDULED_ARRIVAL_AT_PUSHBACK'
SOURCE_FIELDS = [*CATEGORICAL_FEATURES, *NUMERIC_FEATURES]
assert len(SOURCE_FIELDS) == 28

required_columns = list(dict.fromkeys([
    "FlightDate", "Dest", "CRSDepTime", TARGET, *SOURCE_FIELDS,
]))


def prepare(source):
    frame = source[required_columns].copy()
    frame = frame.sort_values(["FlightDate", "CRSDepTime"], kind="stable").reset_index(drop=True)
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype(int)
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].astype(object)
    for column in NUMERIC_FEATURES:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame['MINUTES_TO_SCHEDULED_ARRIVAL_AT_PUSHBACK'] = frame["CRSElapsedTime"] - frame["DepDelay"]
    assert frame['MINUTES_TO_SCHEDULED_ARRIVAL_AT_PUSHBACK'].notna().all()
    return frame


prepared = {year: prepare(source_frames[year]) for year in years}
train = pd.concat([prepared[year] for year in TRAIN_YEARS], ignore_index=True)
train = train.sort_values(["FlightDate", "CRSDepTime"], kind="stable").reset_index(drop=True)
test = prepared[TEST_YEAR]
assert train["FlightDate"].max() < test["FlightDate"].min()

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
transformers = [
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
]
if MARGIN_FEATURE is not None:
    margin_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("spline", SplineTransformer(
            n_knots=5, degree=3, knots="quantile",
            extrapolation="linear", include_bias=False,
        )),
        ("scaler", StandardScaler()),
    ])
    transformers.append(("schedule_margin", margin_pipeline, [MARGIN_FEATURE]))

pipeline = Pipeline([
    ("preprocessor", ColumnTransformer(transformers=transformers, remainder="drop")),
    ("classifier", LogisticRegression(
        solver="liblinear", max_iter=5_000, random_state=RANDOM_STATE,
        C=0.01, l1_ratio=1.0, class_weight=None,
    )),
])
model_columns = [*SOURCE_FIELDS]
if MARGIN_FEATURE is not None:
    model_columns.append(MARGIN_FEATURE)

fit_start = perf_counter()
pipeline.fit(train[model_columns], train[TARGET])
fit_seconds = perf_counter() - fit_start
prepared_predictors = len(pipeline.named_steps["preprocessor"].get_feature_names_out())
predict_start = perf_counter()
probabilities = pipeline.predict_proba(test[model_columns])[:, 1]
predict_seconds = perf_counter() - predict_start

print(f"Training rows: {len(train):,}; delay rate: {train[TARGET].mean():.4f}")
print(f"Source fields: {len(SOURCE_FIELDS)}; prepared predictors: {prepared_predictors}")
print(f"Fit: {fit_seconds:,.1f}s; 2024 prediction: {predict_seconds:,.1f}s")

Training rows: 217,301; delay rate: 0.2249
Source fields: 28; prepared predictors: 108
Fit: 2.7s; 2024 prediction: 0.1s


## Post-test comparison

The official row is copied from the final 2019-only evaluation. The combined-training row uses the same 2024 outcomes and is reported only as a retraining sensitivity.

In [4]:
def ranking_metrics(y_true, probabilities):
    return {
        "average_precision": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "brier_score": brier_score_loss(y_true, probabilities),
    }


def operating_metrics(y_true, probabilities, threshold):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "mcc": matthews_corrcoef(y_true, predictions),
        "predicted_positive_rate": predictions.mean(),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

combined_ranking = ranking_metrics(test[TARGET], probabilities)
combined_default = operating_metrics(test[TARGET], probabilities, 0.50)
combined_frozen = operating_metrics(test[TARGET], probabilities, FROZEN_THRESHOLD)

official_ranking = {'average_precision': 0.8757, 'roc_auc': 0.9254, 'brier_score': 0.0626}
official_frozen = {'threshold': 0.39, 'accuracy': 0.9201, 'balanced_accuracy': 0.8535, 'precision': 0.874, 'recall': 0.7362, 'f1': 0.7992, 'mcc': 0.754}

ranking_comparison = pd.DataFrame([
    {"training": "2019 only — official final", **official_ranking},
    {"training": "2019 + 2023 — post-test", **combined_ranking},
]).set_index("training")
for metric in official_ranking:
    ranking_comparison.loc["2019 + 2023 — post-test", f"{metric}_change"] = (
        combined_ranking[metric] - official_ranking[metric]
    )

operating_results = pd.DataFrame([
    {"training": "2019 only — official final", "threshold_policy": "Frozen", **official_frozen},
    {"training": "2019 + 2023 — post-test", "threshold_policy": "Default", **combined_default},
    {"training": "2019 + 2023 — post-test", "threshold_policy": "Frozen", **combined_frozen},
]).set_index(["training", "threshold_policy"])

display(ranking_comparison)
display(operating_results)

,average_precision,roc_auc,brier_score,average_precision_change,roc_auc_change,brier_score_change
training,,,,,,
2019 only — official final,0.8757,0.9254,0.0626,NaN,NaN,NaN
2019 + 2023 — post-test,0.8769,0.9261,0.0621,0.0012,0.0007,-0.0005


threshold  accuracy  \
training                   threshold_policy                        
2019 only — official final Frozen               0.3900    0.9201   
2019 + 2023 — post-test    Default              0.5000    0.9227   
                           Frozen               0.3900    0.9192   

                                             balanced_accuracy  precision  \
training                   threshold_policy                                 
2019 only — official final Frozen                       0.8535     0.8740   
2019 + 2023 — post-test    Default                      0.8473     0.9083   
                           Frozen                       0.8569     0.8605   

                                             recall     f1    mcc  \
training                   threshold_policy                         
2019 only — official final Frozen            0.7362 0.7992 0.7540   
2019 + 2023 — post-test    Default           0.7145 0.7999 0.7612   
                           Frozen            0.7472 0.7998 0.7525   

                                             predicted_positive_rate  \
training                   threshold_policy                            
2019 only — official final Frozen                                NaN   
2019 + 2023 — post-test    Default                            0.1700   
                           Frozen                             0.1877   

                                                     tn         fp         fn  \
training                   threshold_policy                                     
2019 only — official final Frozen                   NaN        NaN        NaN   
2019 + 2023 — post-test    Default          80,331.0000 1,629.0000 6,450.0000   
                           Frozen           79,222.0000 2,738.0000 5,712.0000   

                                                     tp  
training                   threshold_policy              
2019 only — official final Frozen                   NaN  
2019 + 2023 — post-test    Default          16,145.0000  
                           Frozen           16,883.0000

## Interpretation rule

This experiment changes only the training population. Its result does not replace Logistic Regression 2B-02 as the official selected experiment and does not reopen model or threshold selection.